# Set up your Databricks project
Run only after attaching notebook compute. The project uses Unity Catalog managed storage. It does not connect to your Week 10 SQL databases. This notebook creates only the personal schema and volume below; check their names before running.

In [ ]:
import re
# EDIT CATALOG if the detected default is not a writable Unity Catalog catalog.
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "iman_mining_lab"
VOLUME = "project_files"
for identifier in [CATALOG, SCHEMA, VOLUME]:
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier), "Use simple catalog/schema names"
assert CATALOG not in ["hive_metastore", "system"], "Choose a writable Unity Catalog catalog in CATALOG"
PREFIX = f"`{CATALOG}`.`{SCHEMA}`"
FILES = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
def table(name):
    assert re.fullmatch(r"[a-z_]+", name)
    return f"{PREFIX}.`{name}`"
print("Catalog:", CATALOG, "Schema:", SCHEMA, "Volume path:", FILES)


In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PREFIX}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {PREFIX}.`{VOLUME}`")
print("Upload mining_shifts_raw.csv and next_shifts.csv into:", FILES)


## Stop and upload
In Catalog, open the catalog/schema printed above → Volumes → project_files → Upload. Select the two files from the downloaded data folder. Upload them directly at the volume root, not inside another folder. You do not need to upload the ZIP or notebook files here. Return and run the next cell.

In [ ]:
files = dbutils.fs.ls(FILES)
display(files)
names = {x.name for x in files}
assert {"mining_shifts_raw.csv", "next_shifts.csv"}.issubset(names), "Upload both CSV files at volume root"


In [ ]:
import importlib.metadata as md
for package in ["pandas", "numpy", "scikit-learn", "matplotlib", "mlflow"]:
    try: print(package, md.version(package))
    except md.PackageNotFoundError: print(package, "MISSING: see requirements.txt and the guide")
